<a href="https://colab.research.google.com/github/RasalaPraneeth2006/skill-map-agent/blob/main/skill_map_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -qU langchain langchain-google-genai langchain-tavily

In [ ]:
from google.colab import userdata

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch

import requests

from langchain.tools import tool

print("Imports successful")

Imports successful


In [ ]:
gemini_api_key = userdata.get("GEMINI_API_KEY")
tavily_api_key = userdata.get("TAVILY_API_KEY")
rapidapi_key = userdata.get("RAPIDAPI_KEY")

print("Gemini key:", "Loaded" if gemini_api_key else "NOT FOUND")
print("Tavily key:", "Loaded" if tavily_api_key else "NOT FOUND")
print("RapidAPI key:", "Loaded" if rapidapi_key else "NOT FOUND")

Gemini key: Loaded
Tavily key: Loaded
RapidAPI key: Loaded


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=gemini_api_key,
    temperature=0
)

print("Gemini model created successfully")

Gemini model created successfully


In [ ]:
response = model.invoke(
    "What is Generative AI? Explain in one sentence."
)

print("\nGemini Response:")
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Gemini Response:
[{'type': 'text', 'text': 'Generative AI is a type of artificial intelligence that creates new, original content—such as text, images, audio, and code—by learning patterns from existing data.', 'extras': {'signature': 'EvUNCvINARFNMg8nNZFm7qCd4yjIFyn5PvSfQky4Rl582FobTH7ypsh8Wef7KWBw5MQRkeQnfqeYoVGBvfzapHyFg84dfMhtCAwQ2uOOvKXtUDgvH1LXpTm/JtVowdrT6BJFVIyG0V/3j0+VMJde1xLG3wi4qPgpe+klkF/dUYkh4SkPzVol5SviGIquM1NTmcJ5jdh4dN/DiAXSLv6bHxyI01CqNSsEqs5f4KTY110sWOFxIM9wz46sCWnxvvl+Ea7vsz4xAaBBW4QYizWAHT3oSkASL8/bFhU+v9dnNMpGzUp1yJnURUjgL5/fX9z88SL9QF7JUPpFKtL/0GqzIO1JcxAJW1yB23OVyQ0ZQEUrbvunCc2X8Tl3QPXE2D0WegUEp4FBXBjNJCoPFBi0/tWWO1e50ysz3fCmlN3owSO0J7DE9UeLP7yHN09pT/2t/XsmT1NxMaXXDGHXwXKKHUn9TLnBKRxVqJBQ6S54NOwrxaV7IPK1dHgHljWb797IIxffK53LzKi4f2O3E7BoPdS7AXrofypaWp3uzr6nPBiSTDqHHJCi8A31zPIGjqDDd2rBxRxslat8NEQ6sPxHf7uS4qGibo56LsBF3Uebb4RD7rTHylwKDmgPmv8jib6lvrQQPZRhtrErBzEoOJVSOyEXcBl4teQvMmc8n5bsmhwvV011XMpeEJs+MO5Sr6pOoXV2T7APqBl1tleEUUuHhStNPZcfhqqPEtf+51j0e7D6QHS5dHYl+Ap5QD5

In [ ]:
skill_demand_tool = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced",
    tavily_api_key=tavily_api_key
)

print("\nTavily skill demand tool created successfully")


Tavily skill demand tool created successfully


In [ ]:
result = skill_demand_tool.invoke({
    "query": "Generative AI industry demand career trends salary jobs"
})

print("\nTavily Search Result:")
pprint(result)


Tavily Search Result:
{'answer': None,
 'follow_up_questions': None,
 'images': [],
 'query': 'Generative AI industry demand career trends salary jobs',
 'request_id': 'a23a40b1-2863-4709-8fc4-e96663bfacb5',
 'response_time': 2.4,
 'results': [{'content': 'Generative AI jobs’ salary ranges from a median '
                         'total pay of $154,000 for AI engineers to $132,000 '
                         'for prompt engineers and $110,000 for AI writers [1, '
                         '2, 3].\n'
                         '\n'
                         'The entry-level generative AI salary is $122,000 for '
                         'AI engineers, $110,000 for prompt engineers, and '
                         '$66,000 for AI writers [1, 2, 3]. Learn more about '
                         'the generative AI salary for each career and how '
                         'factors like where you live, the industry and '
                         'company you work for, and your experience level can 

In [ ]:
@tool
def search_jobs(skill: str, location: str) -> list:
    """
    Search for jobs based on skill and location.
    """

    print("\nCalling search jobs tool")
    print(f"Searching for {skill} jobs in {location}")

    url = "https://jsearch.p.rapidapi.com/search-v2"

    querystring = {
        "query": f"{skill} jobs in {location}",
        "num_pages": "1",
        "country": "in",
        "employment_types": "FULLTIME,INTERN",
        "job_requirements": "under_3_years_experience,no_experience"
    }

    headers = {
        "x-rapidapi-key": rapidapi_key,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }

    response = requests.get(
        url,
        headers=headers,
        params=querystring,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    raw_jobs = data.get("data", [])

    # Make sure we have a list
    if not isinstance(raw_jobs, list):
        print("Unexpected job data format")
        return []

    print(f"Found {len(raw_jobs)} jobs")

    result = []

    for job in raw_jobs:

        # Ignore invalid entries
        if not isinstance(job, dict):
            continue

        result.append({
            "title": job.get("job_title", ""),
            "company": job.get("employer_name", ""),
            "location": (
                job.get("job_city")
                or job.get("job_location")
                or location
            ),
            "apply_link": job.get("job_apply_link", "")
        })

    return result


print("\nJob search tool created successfully")


Job search tool created successfully


In [ ]:
jobs = search_jobs.invoke({
    "skill": "Generative AI",
    "location": "Hyderabad"
})

print("\nJob Search Results:")
pprint(jobs)


Calling search jobs tool
Searching for Generative AI jobs in Hyderabad
Unexpected job data format

Job Search Results:
[]


In [ ]:
print("\n" + "=" * 60)
print("GENERATIVE AI JOBS IN HYDERABAD")
print("=" * 60)

if jobs:

    for i, job in enumerate(jobs, 1):

        print(f"\nJOB {i}")
        print("-" * 40)
        print(f"Title    : {job['title']}")
        print(f"Company  : {job['company']}")
        print(f"Location : {job['location']}")
        print(f"Apply    : {job['apply_link']}")

else:

    print("No jobs found.")



GENERATIVE AI JOBS IN HYDERABAD
No jobs found.


In [ ]:
agent = create_agent(
    model=model,
    tools=[
        skill_demand_tool,
        search_jobs
    ],
    system_prompt="""
You are a Skill-to-Career Mapping assistant.

You help students understand technology demand,
career opportunities and job opportunities.

You have two tools:

1. skill_demand_tool
   Use it to research:
   - Industry demand
   - Career trends
   - Salary information
   - Required skills
   - Technology trends

2. search_jobs
   Use it to find actual job listings based on:
   - Skill
   - Location

IMPORTANT OUTPUT FORMAT:

When the user asks about skill demand, answer using:

==================================================
SKILL DEMAND
==================================================

Skill:
Industry Demand:
Career Opportunities:
Important Skills:
Salary Information:
Future Scope:

==================================================


When the user asks for jobs, answer using:

==================================================
JOB OPPORTUNITIES
==================================================

Job 1
--------------------------------------------------
Job Title:
Company:
Location:
Application Link:

Job 2
--------------------------------------------------
Job Title:
Company:
Location:
Application Link:

Job 3
--------------------------------------------------
Job Title:
Company:
Location:
Application Link:

==================================================

RULES:

- Use actual information returned by the tools.
- Do not invent jobs.
- Do not invent companies.
- Do not invent application links.
- Keep the output clean and readable.
- Do not show internal tool execution details.
- Do not show Python code.
"""
)

print("\nAI Agent created successfully")


AI Agent created successfully


In [ ]:
user_query = "What's the current demand for Generative AI?"

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_query
        }
    ]
})

print("\n" + "=" * 60)
print("AGENT RESPONSE")
print("=" * 60)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



AGENT RESPONSE
[{'type': 'text', 'text': '==================================================\nSKILL DEMAND\n==================================================\n\nSkill: Generative AI\n\nIndustry Demand:\nExtremely High & Rapidly Growing. Job postings requiring Generative AI skills have surged exponentially, growing over 3.5 times faster than traditional job listings. Enterprise adoption is expanding beyond big tech into consulting, finance, healthcare, and retail, with 88% of organizations actively integrating AI capabilities into their core business functions.\n\nCareer Opportunities:\n- Generative AI Engineer\n- Machine Learning Engineer\n- AI Research Scientist\n- LLM / Prompt Engineer\n- AI Solutions Architect\n- AI Product Manager\n- MLOps & Enterprise AI Architect\n\nImportant Skills:\n- Programming: Python, C++, SQL\n- AI Frameworks & Tooling: PyTorch, TensorFlow, Hugging Face, LangChain, LlamaIndex\n- Core Techniques: Large Language Models (LLMs), Fine-Tuning, Retrieval-Augmen

In [ ]:
user_query = """
Find Generative AI jobs for freshers in Hyderabad.
Give me the job title, company, location and application link.
"""

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_query
        }
    ]
})

print("\n" + "=" * 60)
print("JOB SEARCH AGENT RESPONSE")
print("=" * 60)

print(response["messages"][-1].content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Calling search jobs tool
Searching for Generative AI jobs in Hyderabad
Unexpected job data format


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/l


JOB SEARCH AGENT RESPONSE
[{'type': 'text', 'text': '### Industry Demand & Skill Overview for Generative AI\n\n#### Why Generative AI is Important\nGenerative AI and Large Language Models (LLMs) are transforming how software applications are architected, deployed, and interacted with across modern tech companies. Hyderabad has emerged as a premier IT hub in India, with both MNCs and innovative startups actively recruiting fresh graduates to build AI agents, Retrieval-Augmented Generation (RAG) systems, and conversational AI platforms.\n\n#### Key Career Roles for Freshers\n* **Generative AI Engineer / Trainee**\n* **AI/ML Trainee**\n* **AI-Native Software Developer**\n* **LLM Application Developer**\n\n#### Essential Skills & Technologies\n* **Programming Languages:** Python, JavaScript/TypeScript\n* **Frameworks & Orchestration:** LangChain, LlamaIndex, Transformers (Hugging Face), PyTorch\n* **AI Concepts:** Prompt Engineering, RAG (Retrieval-Augmented Generation), Fine-Tuning, AI A